<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Sıfırdan Byte Pair Encoding (BPE) Tokenizer

- Bu, GPT-2'den GPT-4'e, Llama 3 gibi modellerde kullanılan popüler byte pair encoding (BPE) token'lara ayırma algoritmasını eğitim amacıyla sıfırdan uygulayan bağımsız bir not defteridir
- Token'lara ayırmanın amacı hakkında daha fazla ayrıntı için lütfen [Bölüm 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb) dosyasına bakın; buradaki kod BPE algoritmasını açıklayan bonus materyaldir
- OpenAI'ın orijinal GPT modellerini eğitmek için uyguladığı özgün BPE tokenizer'ına [buradan](https://github.com/openai/gpt-2/blob/master/src/encoder.py) ulaşabilirsiniz
- BPE algoritması ilk kez 1994'te tanımlanmıştır: Philip Gage, "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- Llama 3 dahil çoğu proje bugünlerde hesaplama başarımı nedeniyle OpenAI'ın açık kaynaklı [tiktoken kütüphanesini](https://github.com/openai/tiktoken) kullanıyor; bu kütüphane örneğin önceden eğitilmiş GPT-2 ve GPT-4 tokenizer'larını yüklemeye olanak tanır (Llama 3 modelleri de GPT-4 tokenizer'ı kullanılarak eğitilmiştir)
- Yukarıdaki uygulamalarla bu not defterindeki uygulamam arasındaki fark, buradakinin (eğitim amacıyla) tokenizer'ı eğitmek için bir fonksiyon da içermesidir
- Eğitim desteği olan ve muhtemelen daha başarımlı olan [minBPE](https://github.com/karpathy/minbpe) adlı bir uygulama da var (buradaki uygulamam eğitim amacına odaklıdır); `minbpe`'den farklı olarak benim uygulamam ayrıca orijinal OpenAI tokenizer sözlüğünü ve BPE "birleştirmelerini" (merges) yüklemeye de izin verir (ayrıca Hugging Face tokenizer'ları da çeşitli tokenizer'ları eğitip yükleyebilir; daha fazla bilgi için Nepalce üzerinde bir BPE tokenizer'ı eğiten bir okurun [bu GitHub tartışmasına](https://github.com/rasbt/LLMs-from-scratch/discussions/485) bakın)

&nbsp;
# 1. Byte pair encoding'in (BPE) arkasındaki temel fikir

- BPE'deki temel fikir, LLM eğitimi için metni bir tam sayı temsiline (token kimlikleri) dönüştürmektir (bkz. [Bölüm 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 Bit'ler ve bayt'lar

- BPE algoritmasına geçmeden önce bayt (byte) kavramını tanıtalım
- Metni bir bayt dizisine dönüştürmeyi düşünün (nihayetinde BPE, "byte" pair encoding'in kısaltması):

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- Bir `bytearray` nesnesi üzerinde `list()` çağırdığımızda her bayt ayrı bir eleman olarak ele alınır ve sonuç, bayt değerlerine karşılık gelen bir tam sayı listesi olur:

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- Bu, metni bir LLM'in gömme katmanı için ihtiyaç duyduğumuz token kimliği temsiline dönüştürmenin geçerli bir yolu olurdu
- Ancak bu yaklaşımın dezavantajı, her karakter için bir kimlik oluşturmasıdır (kısa bir metin için bile bu çok fazla kimlik demek!)
- Yani 17 karakterlik bir girdi metni için LLM'e girdi olarak 17 token kimliği kullanmamız gerekir:

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- Daha önce LLM'lerle çalıştıysanız, BPE tokenizer'larının her karakter yerine tam kelimeler veya alt kelimeler için token kimliği içeren bir sözlüğe sahip olduğunu biliyor olabilirsiniz
- Örneğin GPT-2 tokenizer'ı aynı metni ("This is some text") 17 yerine yalnızca 4 token'a ayırır: `1212, 318, 617, 2420`
- Bunu etkileşimli [tiktoken uygulamasını](https://tiktokenizer.vercel.app/?model=gpt2) veya [tiktoken kütüphanesini](https://github.com/openai/tiktoken) kullanarak doğrulayabilirsiniz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- Bir bayt 8 bit'ten oluştuğu için, tek bir baytın temsil edebileceği 2<sup>8</sup> = 256 olası değer vardır; bunlar 0 ile 255 arasındadır
- Bunu `bytearray(range(0, 257))` kodunu çalıştırarak doğrulayabilirsiniz; bu kod size `ValueError: byte must be in range(0, 256)` uyarısını verecektir
- Bir BPE tokenizer'ı genellikle bu 256 değeri ilk 256 tek karakterli token'ı olarak kullanır; bunu şu kodu çalıştırarak gözle kontrol edebilirsiniz:

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```

- Yukarıda, 256 ve 257 numaralı kayıtların tek karakterli değil çift karakterli değerler olduğuna dikkat edin (bir boşluk + bir harf); bu, orijinal GPT-2 BPE tokenizer'ının küçük bir eksikliğidir (GPT-4 tokenizer'ında düzeltilmiştir)

&nbsp;
## 1.2 Sözlüğü oluşturmak

- BPE token'lara ayırma algoritmasının amacı, `298: ent` gibi sık geçen alt kelimelerden (örneğin *entangle, entertain, enter, entrance, entity, ...* içinde bulunabilir) ve hatta tam kelimelerden oluşan bir sözlük oluşturmaktır:

```
318: is
617: some
1212: This
2420: text
```

- BPE algoritması ilk kez 1994'te tanımlanmıştır: Philip Gage, "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- Asıl kod uygulamasına geçmeden önce, bugün LLM tokenizer'larında kullanılan biçim aşağıdaki bölümlerde anlatıldığı gibi özetlenebilir.

&nbsp;
## 1.3 BPE algoritmasının ana hatları

**1. Sık geçen çiftleri belirle**
- Her yinelemede metni tarayarak en sık geçen bayt (veya karakter) çiftini bul

**2. Değiştir ve kaydet**

- Bu çifti, henüz kullanılmayan yeni bir yer tutucu kimlikle değiştir (ör. 0...255 ile başlıyorsak ilk yer tutucu 256 olur)
- Bu eşlemeyi bir arama tablosuna kaydet
- Arama tablosunun boyutu bir hiperparametredir; buna "sözlük boyutu" (vocabulary size) da denir (GPT-2 için bu 50.257'dir)

**3. Kazanç kalmayana kadar tekrarla**

- 1. ve 2. adımları, en sık geçen çiftleri sürekli birleştirerek tekrarla
- Daha fazla sıkıştırma mümkün olmadığında dur (ör. hiçbir çift birden fazla geçmiyorsa)

**Açma (kod çözme)**

- Orijinal metni geri getirmek için, arama tablosunu kullanarak her kimliği karşılık gelen çiftle değiştirip süreci tersine çevir



&nbsp;
## 1.4 BPE algoritması örneği

### 1.4.1 Kodlama kısmının somut örneği (1.3 bölümündeki 1. ve 2. adımlar)

- Elimizde, bir BPE tokenizer'ı için sözlük oluşturmak istediğimiz `the cat in the hat` metni (eğitim veri kümesi) olduğunu varsayalım

**1. Yineleme**

1. Sık geçen çiftleri belirle
  - Bu metinde "th" iki kez geçiyor (başta ve ikinci "e"den önce)

2. Değiştir ve kaydet
  - "th" ifadesini henüz kullanılmayan yeni bir token kimliğiyle, ör. 256 ile değiştir
  - yeni metin: `<256>e cat in <256>e hat`
  - yeni sözlük:

```
  0: ...
  ...
  256: "th"
```

**2. Yineleme**

1. **Sık geçen çiftleri belirle**  
   - `<256>e cat in <256>e hat` metninde `<256>e` çifti iki kez geçiyor

2. **Değiştir ve kaydet**  
   - `<256>e` ifadesini henüz kullanılmayan yeni bir token kimliğiyle, örneğin `257` ile değiştir.  
   - Yeni metin:
     ```
     <257> cat in <257> hat
     ```
   - Güncellenen sözlük:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**3. Yineleme**

1. **Sık geçen çiftleri belirle**  
   - `<257> cat in <257> hat` metninde `<257> ` çifti iki kez geçiyor (biri başta, biri "hat" öncesinde).

2. **Değiştir ve kaydet**  
   - `<257> ` ifadesini henüz kullanılmayan yeni bir token kimliğiyle, örneğin `258` ile değiştir.  
   - yeni metin:
     ```
     <258>cat in <258>hat
     ```
   - Güncellenen sözlük:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- ve bu böyle devam eder

&nbsp;
### 1.4.2 Kod çözme kısmının somut örneği (1.3 bölümündeki 3. adım)

- Orijinal metni geri getirmek için, her token kimliğini karşılık gelen çiftle, eklendikleri sıranın tersinden başlayarak değiştirip süreci tersine çeviririz
- Son sıkıştırılmış metinle başla: `<258>cat in <258>hat`
-  `<258>` → `<257> ` yerine koy: `<257> cat in <257> hat`  
- `<257>` → `<256>e` yerine koy: `<256>e cat in <256>e hat`
- `<256>` → "th" yerine koy: `the cat in the hat`

&nbsp;
## 2. Basit bir BPE uygulaması

- Aşağıda, yukarıda anlatılan bu algoritmanın `tiktoken` Python arayüzünü taklit eden bir Python sınıfı olarak uygulaması yer alıyor
- Yukarıdaki kodlama kısmının `train()` üzerinden orijinal eğitim adımını anlattığını unutmayın; ancak `encode()` metodu da benzer şekilde çalışır (özel token'ların ele alınması nedeniyle biraz daha karmaşık görünse de):

1. Girdi metnini tek tek baytlara böl
2. Öğrenilen BPE birleştirmelerinden herhangi biriyle eşleşen komşu token'ları (çiftleri) tekrar tekrar bul ve değiştir (birleştir) — en yüksek "sıradan" (rank) en düşüğe doğru, yani öğrenilme sırasına göre
3. Uygulanabilecek birleştirme kalmayana kadar birleştirmeye devam et
4. Elde edilen nihai token kimliği listesi kodlanmış çıktıdır

In [4]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # token_id -> token_str eşlemesi (ör. {11246: "some"})
        self.vocab = {}
        # token_str -> token_id eşlemesi (ör. {"some": 11246})
        self.inverse_vocab = {}
        # BPE birleştirmeleri sözlüğü: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # Resmî OpenAI GPT-2 birleştirmeleri için bir sıra (rank) sözlüğü kullanın:
        #  {(string_A, string_B): rank} biçiminde; düşük sıra = yüksek öncelik
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Eğitim metnini encode() ile aynı sınır kurallarını kullanarak ön token'lara ayır
        tokens = self.pretokenize_text(text)

        # Sözlüğü benzersiz karakterlerle başlat, varsa "Ġ" dahil
        # İlk 256 ASCII karakteriyle başla
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # İzin verilen özel token'ları ekle
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Her ön token'ı karakter kimliklerine ayır
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE 1-3. adımlar: Sık geçen çiftleri tekrar tekrar bul ve değiştir
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Sözlüğü birleştirilmiş token'larla oluştur
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # Sözlüğü yükle
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json {token_str: id} biçiminde; biz id->str ve str->id istiyoruz
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
    
        # GPT-2'nin yazdırılabilir satır sonu karakteri 'Ċ' (U+010A) 198 kimliğinde bulunmalı
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # 50256 kimliğinde <|endoftext|> bulunmalı
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
    
        # Kolaylık olsun diye '\n' -> 198 için bir takma ad tanımla
        # BPE birleştirmelerinin çalışmaya devam etmesi için yazdırılabilir 'Ċ' karakterini sözlükte tut
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # GPT-2 birleştirmelerini yükle ve sıraları sakla
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
    
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Sembolleri sözlükte olmayan çiftleri atlamak güvenlidir
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.
    
        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.
    
        Returns:
            List of token IDs.
        """
    
        # ---- Bu bölüm, izin verilen özel token'lar açısından tiktoken'ı taklit eder ----
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            # Hiçbirine izin verilmiyor
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            # Belirli bazı token'lara izin veriliyor (ör. <|endoftext|> için bunu kullanıyoruz)
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        # -----------------------------------------------------------------------------

        token_ids = []
        # Bazı özel token'lara izin veriliyorsa, onların etrafından böl ve kimliklerini olduğu gibi geçir
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"
    
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally
    
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
    
            text = text[last_index:]  # remainder to process normally
    
            # Geriye kalan diğer özel değişmezler (literal) için ek koruma
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

    
        # ---- Satır sonu ve satır başı karakterlerinin işlenmesi ----
        tokens = self.pretokenize_text(text)
        # ---------------------------------------------------------------
    
        # Token'ları kimliklere eşle (gerekirse BPE ile)
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
    
        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Token'ı tek tek karakterlere ayır (başlangıç token kimlikleri olarak)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # OpenAI'ın GPT-2 birleştirmelerini yüklemediysek kendi yaklaşımımı kullan
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # Eğitim amaçlı olarak yorumu kaldırabilirsiniz:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # Aksi hâlde, sıraları kullanarak GPT-2 tarzı birleştirme yap:
        # 1) Her kimlik için token_ids değerlerini tekrar "sembol" dizelerine çevir
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # En düşük sıralı çiftin tüm geçtiği yerleri tekrar tekrar birleştir
        while True:
            # Tüm komşu çiftleri topla
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # En iyi (en düşük) sıraya sahip çifti bul
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # Geçerli, sıralanmış bir çift kalmadıysa işimiz bitti
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # O çiftin tüm geçtiği yerleri birleştir
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # i konumunda (first, second) görürsek onları birleştir
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # Son olarak, birleştirilmiş sembolleri tekrar kimliklere çevir
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # GPT-2 özel karakterlerini gerçek karakterlere geri eşle
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Sözlüğü kaydet
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # BPE birleştirmelerini sözlük listesi olarak kaydet
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Sözlüğü yükle
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # BPE birleştirmelerini yükle
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Satır sonu içermeyen normal parça:
            # - Bir kelimeden önce boşluk varsa, ilk kelimenin başına 'Ġ' ekle ve
            #   fazladan boşluklar için tek başına 'Ġ' ekle
            # - Parçanın sonunda boşluk varsa (ör. satır sonundan önce)
            #   tek başına 'Ġ' token'ları ekle (tiktoken 'Ġ' için 220 kimliğini üretir)
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word) # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Sondaki boşluklar (ardından kelime yok): tek başına 'Ġ' token'ları ekle
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequences = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    # Çiftin 2. token'ını kaldır, 1. zaten kaldırılmıştı
                    dq.popleft()
                else:
                    replaced.append(current)

            replaced_sequences.append(replaced)

        return replaced_sequences

- Yukarıdaki `BPETokenizerSimple` sınıfında epey kod var ve bunu ayrıntılı tartışmak bu not defterinin kapsamı dışında; ancak bir sonraki bölüm, sınıf metotlarını biraz daha iyi anlamak için kullanıma dair kısa bir genel bakış sunuyor

## 3. BPE uygulamasının adım adım incelenmesi

- Pratikte [tiktoken](https://github.com/openai/tiktoken) kullanmanızı şiddetle öneririm; çünkü yukarıdaki uygulamam başarıma değil, okunabilirliğe ve eğitim amacına odaklıdır
- Yine de kullanımı aşağı yukarı tiktoken'a benzer; tek fark tiktoken'ın bir eğitim metodu olmamasıdır
- Yukarıdaki `BPETokenizerSimple` Python kodumun nasıl çalıştığını aşağıdaki örneklere bakarak görelim (ayrıntılı kod tartışması bu not defterinin kapsamı dışındadır)

### 3.1 Eğitim, kodlama ve kod çözme

- Önce eğitim veri kümemiz olarak bir örnek metin ele alalım:

In [5]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path


verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

the-verdict.txt already exists in ../01_main-chapter-code/the-verdict.txt


- Ardından BPE tokenizer'ını 1.000 sözlük boyutuyla başlatıp eğitelim
- Daha önce ele alınan bayt değerleri nedeniyle sözlük boyutunun varsayılan olarak zaten 256 olduğunu unutmayın; yani yalnızca 744 sözlük kaydını "öğreniyoruz" (`<|endoftext|>` özel token'ını ve `Ġ` boşluk token'ını da sayarsak, tam olarak 742)
- Karşılaştırma için: GPT-2 sözlüğü 50.257 token, GPT-4 sözlüğü 100.256 token (tiktoken'da `cl100k_base`), GPT-4o ise 199.997 token (tiktoken'da `o200k_base`) içerir; hepsinin eğitim kümeleri yukarıdaki basit örnek metnimize kıyasla çok daha büyüktür

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- Sözlük içeriğini incelemek isteyebilirsiniz (ancak bunun uzun bir liste oluşturacağını unutmayın)

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- Bu sözlük 742 kez birleştirme yapılarak oluşturulmuştur (`= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742`)

In [8]:
print(len(tokenizer.bpe_merges))

742


- Bu, ilk 256 kaydın tek karakterli token'lar olduğu anlamına gelir

- Şimdi, oluşturulan birleştirmeleri `encode` metodu üzerinden kullanarak bir metni kodlayalım:

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [11]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 56
Number of token IDs: 22


- Yukarıdaki uzunluklardan görebiliyoruz ki 42 karakterlik bir cümle 20 token kimliğine kodlandı; bu, karakter-bayt tabanlı bir kodlamaya kıyasla girdi uzunluğunu kabaca yarıya indiriyor

- Sözlüğün kendisinin `decode()` metodunda kullanıldığını ve token kimliklerini tekrar metne eşlememizi sağladığını unutmayın:

In [12]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [13]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


- Her token kimliği üzerinde tek tek dolaşmak, token kimliklerinin sözlük aracılığıyla nasıl çözüldüğünü daha iyi anlamamızı sağlar:

In [14]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .
257 -> <|endoftext|>
256 ->  


- Görüldüğü gibi token kimliklerinin çoğu 2 karakterli alt kelimeleri temsil ediyor; bunun nedeni eğitim verisi metninin çok kısa olması, çok fazla tekrar eden kelime içermemesi ve nispeten küçük bir sözlük boyutu kullanmamızdır

- Özetle, `decode(encode())` çağrısı herhangi bir girdi metnini yeniden üretebilmelidir:

In [15]:
tokenizer.decode(
    tokenizer.encode("This is some text.")
)

'This is some text.'

In [16]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.2 Tokenizer'ı kaydetmek ve yüklemek

- Şimdi, eğitilmiş tokenizer'ı daha sonra yeniden kullanmak üzere nasıl kaydedebileceğimize bakalım:

In [17]:
# Eğitilmiş tokenizer'ı kaydet
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [18]:
# Tokenizer'ı yükle
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- Yüklenen tokenizer, öncekiyle aynı sonuçları üretebilmelidir:

In [19]:
print(tokenizer2.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


In [20]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

&nbsp;
### 3.3 OpenAI'ın orijinal GPT-2 BPE tokenizer'ını yüklemek

- Son olarak, OpenAI'ın GPT-2 tokenizer dosyalarını yükleyelim

In [21]:
# Bu dizinde yoksa dosyaları indir

# Aranacak dizinleri ve indirilecek dosyaları tanımla
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Dizinlerin var olduğundan emin ol ve gerekirse dosyaları indir
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

vocab.bpe already exists in ../02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
encoder.json already exists in ../02_bonus_bytepair-encoder/gpt2_model/encoder.json


- Ardından dosyaları `load_vocab_and_merges_from_openai` metodu aracılığıyla yüklüyoruz:

In [22]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- Sözlük boyutu `50257` olmalı; aşağıdaki kodla bunu doğrulayabiliriz:

In [23]:
len(tokenizer_gpt2.vocab)

50257

- Artık GPT-2 tokenizer'ını `BPETokenizerSimple` nesnemiz üzerinden kullanabiliriz:

In [24]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

[1212, 318, 617, 2420]


In [25]:
print(tokenizer_gpt2.decode(token_ids))

This is some text


- Bunun doğru token'ları ürettiğini, etkileşimli [tiktoken uygulamasını](https://tiktokenizer.vercel.app/?model=gpt2) veya [tiktoken kütüphanesini](https://github.com/openai/tiktoken) kullanarak doğrulayabilirsiniz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```



&nbsp;
# 4. Sonuç

- İşte bu kadar! BPE özetle böyle çalışır; üstelik yeni tokenizer'lar oluşturmak ya da orijinal OpenAI GPT-2 modelinden GPT-2 tokenizer sözlüğünü ve birleştirmelerini yüklemek için bir eğitim metoduyla birlikte
- Umarım bu kısa öğreticiyi eğitim amacıyla faydalı bulmuşsunuzdur; sorularınız varsa [buradan](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a) yeni bir Discussion açmaktan çekinmeyin
- Diğer tokenizer uygulamalarıyla başarım karşılaştırması için lütfen [bu not defterine](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) bakın